# NeerNiti — Phase 6.0 Production Blueprint (Uttar Pradesh)
### SAR Physics, Exact Haversine Purging, TWI/River Hydrology, Optuna & Voting Ensemble

This notebook implements the "Paper 8 & 10" methodologies without spatial leakage:
1. **Physical Hydrology:** Injects Topographic Wetness Index (TWI) and Distance to Rivers.
2. **Optuna Spatial Metaheuristics:** Finds optimal hyperparameters evaluated strictly on Haversine purged folds.
3. **Leak-Free Voting Ensemble:** Averages XGBoost, LightGBM, and CatBoost probabilities to safely boost AUC.

In [ ]:
!pip install -q earthengine-api xgboost lightgbm catboost scikit-learn pandas numpy matplotlib tqdm google-api-python-client seaborn optuna

import os, json, time, math, warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna
from optuna.samplers import TPESampler
import joblib
from sklearn.neighbors import BallTree
from sklearn.model_selection import KFold
from sklearn.ensemble import VotingClassifier
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    accuracy_score, confusion_matrix, roc_curve, classification_report,
    matthews_corrcoef, balanced_accuracy_score, precision_recall_curve, auc
)
from datetime import datetime, timezone
import ee
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
RUNTIME = 'kaggle' if os.path.exists('/kaggle') else 'colab'
print(f'Runtime: {RUNTIME.upper()}')

In [ ]:
if RUNTIME == 'kaggle':
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    creds = service_account.Credentials.from_service_account_info(
        json.loads(_secrets.get_secret('GEE_SERVICE_ACCOUNT_KEY')),
        scopes=['https://www.googleapis.com/auth/earthengine', 'https://www.googleapis.com/auth/drive']
    )
    ee.Initialize(creds, project=_secrets.get_secret('GEE_PROJECT_ID'))
    drive_service = build('drive', 'v3', credentials=creds)
else:
    ee.Authenticate()
    ee.Initialize(project=input('Enter GEE Project ID: ').strip())
print('GEE Authenticated.')

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

TEL_2021_CSV_PATH = '/kaggle/input/datasets/chinmay302/neerniti/gwl_tel_6_hourly_upgw_up_2021_2025.csv'
TEL_2026_CSV_PATH = '/kaggle/input/datasets/chinmay302/neerniti/gwl_tel_6_hourly_upgw_up_2026_2030.csv'

# New cache file to force re-extraction with River data
GEE_CACHE_FILE = '/kaggle/working/up_gee_cache_v6_0.csv'
GEE_BUFFER_M = 500
SPATIAL_BLOCK_DEGREES = 0.25
SPATIAL_BUFFER_KM = 3.0
VERSION_ID = f'neerniti_up_v6_0_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}'

print(f'Output: {VERSION_ID}')

In [ ]:
def process_telemetry():
    print("Loading UP Telemetry datasets (Strict Chronological Pairing)...")
    def load_and_pair(path, label):
        try:
            df = pd.read_csv(path, low_memory=False)
            df['Date'] = pd.to_datetime(df['Data Acquisition Time'], errors='coerce', format='mixed', dayfirst=True)
            df = df.dropna(subset=['Date'])
            df['Month'] = df['Date'].dt.month
            df['Year'] = df['Date'].dt.year
            lvl_col = [c for c in df.columns if 'meter' in c.lower()][0]
            df[lvl_col] = pd.to_numeric(df[lvl_col], errors='coerce').abs()
            
            pre = df[df['Month'].isin([4, 5])].groupby(['Latitude', 'Longitude', 'Year', 'District'])[lvl_col].mean().reset_index()
            pre.rename(columns={lvl_col: 'pre_monsoon_m'}, inplace=True)
            post = df[df['Month'].isin([10, 11])].groupby(['Latitude', 'Longitude', 'Year', 'District'])[lvl_col].mean().reset_index()
            post.rename(columns={lvl_col: 'post_monsoon_m'}, inplace=True)
            return pd.merge(pre, post, on=['Latitude', 'Longitude', 'Year', 'District'], how='inner')
        except Exception as e: return pd.DataFrame()

    tel1 = load_and_pair(TEL_2021_CSV_PATH, '2021-2025')
    tel2 = load_and_pair(TEL_2026_CSV_PATH, '2026-2030')
    
    combined = pd.concat([tel1, tel2], ignore_index=True)
    final_df = combined.groupby(['Latitude', 'Longitude']).agg({'pre_monsoon_m': 'mean', 'post_monsoon_m': 'mean', 'District': 'first'}).reset_index()
    
    final_df['recharge_potential'] = final_df['pre_monsoon_m'] - final_df['post_monsoon_m']
    final_df['target'] = ((final_df['pre_monsoon_m'] >= 6.0) & (final_df['pre_monsoon_m'] <= 35.0) & (final_df['recharge_potential'] >= 1.5)).astype(int)
    
    final_df['well_id'] = ['UP_WELL_' + str(i) for i in range(len(final_df))]
    final_df['spatial_block_id'] = (np.floor(final_df['Latitude'] / SPATIAL_BLOCK_DEGREES).astype(str) + '_' + np.floor(final_df['Longitude'] / SPATIAL_BLOCK_DEGREES).astype(str))
    
    print(f"Aggregated dataset: {len(final_df)} wells. Positive: {final_df['target'].sum()} | Negative: {len(final_df)-final_df['target'].sum()}")
    return final_df

dataset = process_telemetry()

In [ ]:
# Derived from UP Groundwater Year Books
bhabar_terai = ['saharanpur', 'bijnor', 'moradabad', 'rampur', 'bareilly', 'pilibhit', 'lakhimpur', 'kheri', 'bahraich', 'shravasti', 'balrampur', 'siddharthnagar', 'mahrajganj']
bundelkhand_hard_rock = ['jhansi', 'lalitpur', 'jalaun', 'hamirpur', 'mahoba', 'banda', 'chitrakoot', 'mirzapur', 'sonbhadra']

def get_hydro_zone(district):
    d = str(district).strip().lower()
    if d in bhabar_terai: return 0
    if d in bundelkhand_hard_rock: return 2
    return 1

dataset['pdf_hydro_zone'] = dataset['District'].apply(get_hydro_zone)
print(f'Geological Zone Distribution: {dataset["pdf_hydro_zone"].value_counts().to_dict()}')

In [ ]:
if not os.path.exists(GEE_CACHE_FILE):
    print('Initiating chunked extraction (including Rivers)...')
    
    dem = ee.ImageCollection('COPERNICUS/DEM/GLO30_2024_1').mosaic().select(0)
    slope = ee.Terrain.slope(dem).rename('slope_deg')
    curvature = dem.convolve(ee.Kernel.laplacian8()).rename('curvature')
    flow_acc = ee.Image('WWF/HydroSHEDS/15ACC').rename('flow_accumulation')
    clay = (ee.Image('projects/soilgrids-isric/clay_mean').select('clay_0-5cm_mean').divide(10).rename('soil_clay_pct'))
    chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterDate('2021-01-01', '2025-01-01').filter(ee.Filter.calendarRange(6, 9, 'month')).sum().divide(4).rename('monsoon_rainfall_mm')
    grace = ee.ImageCollection('NASA/GRACE/MASS_GRIDS_V04/MASCON_CRI').filterDate('2021-01-01', '2025-01-01').select('lwe_thickness').mean().rename('grace_gw_storage_cm')
    
    # Distance to Free Flowing Rivers (New Feature!)
    rivers = ee.FeatureCollection('WWF/HydroSHEDS/v1/FreeFlowingRivers')
    dist_to_river = ee.Image().paint(rivers, 1).unmask(0).fastDistanceTransform(1024).sqrt().multiply(30).rename('distance_to_river_m')
    
    base_stack = ee.Image.cat([slope, curvature, flow_acc, clay, chirps, grace, dem.rename('elevation_m'), dist_to_river])
    
    chunk_size = 50
    rows = []
    
    for i in range(0, len(dataset), chunk_size):
        chunk_df = dataset.iloc[i:i+chunk_size]
        features = [ee.Feature(ee.Geometry.Point([row['Longitude'], row['Latitude']]), {'well_id': str(row['well_id'])}) for _, row in chunk_df.iterrows()]
        buffered_fc = ee.FeatureCollection(features).map(lambda f: f.buffer(GEE_BUFFER_M))
        
        s1 = (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(buffered_fc.geometry())
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
              .filter(ee.Filter.eq('instrumentMode', 'IW')))
        delta_sar = s1.filter(ee.Filter.calendarRange(10, 11, 'month')).median().select('VV').subtract(s1.filter(ee.Filter.calendarRange(10, 11, 'month')).median().select('VH')).subtract(
            s1.filter(ee.Filter.calendarRange(4, 5, 'month')).median().select('VV').subtract(s1.filter(ee.Filter.calendarRange(4, 5, 'month')).median().select('VH'))
        ).rename('delta_sar_vv_vh')
        
        s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(buffered_fc.geometry()).filterDate('2021-01-01', '2022-01-01')
        delta_ndwi = s2.filter(ee.Filter.calendarRange(10, 11, 'month')).median().normalizedDifference(['B3', 'B8']).subtract(
            s2.filter(ee.Filter.calendarRange(4, 5, 'month')).median().normalizedDifference(['B3', 'B8'])).rename('delta_ndwi')
        
        stack = ee.Image.cat([base_stack, delta_sar, delta_ndwi])
        results = stack.reduceRegions(collection=buffered_fc, reducer=ee.Reducer.mean(), scale=30, tileScale=4).getInfo()
        
        for f in results['features']: rows.append(f['properties'])
        print(f"  Processed {min(i+chunk_size, len(dataset))}/{len(dataset)}", end='\r')
        
    gee_df = pd.DataFrame(rows)
    gee_df.to_csv(GEE_CACHE_FILE, index=False)
    print(f'\nSuccessfully cached to {GEE_CACHE_FILE}')

gee_features = pd.read_csv(GEE_CACHE_FILE)
dataset['well_id'] = dataset['well_id'].astype(str)
gee_features['well_id'] = gee_features['well_id'].astype(str)
dataset = dataset.merge(gee_features, on='well_id', how='inner')
print(f"Merged {len(dataset)} points with GEE features.")


In [ ]:
# Mathematically compute Topographic Wetness Index (TWI)
dataset['TWI'] = np.log((dataset['flow_accumulation'] + 1) / (np.tan(np.radians(dataset['slope_deg'])) + 0.001))
print("Computed TWI successfully.")

In [ ]:
def generate_purged_folds(df, block_col, coords_cols, n_splits=5, buffer_km=3.0):
    unique_blocks = df[block_col].unique()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    folds = []
    coords_rad = np.radians(df[coords_cols].values)
    buffer_rad = buffer_km / 6371.0
    
    for train_b_idx, test_b_idx in kf.split(unique_blocks):
        train_blocks = unique_blocks[train_b_idx]
        test_blocks = unique_blocks[test_b_idx]
        test_idx = df[df[block_col].isin(test_blocks)].index
        if len(test_idx) == 0: continue
        test_coords = coords_rad[test_idx]
        train_candidates = df[df[block_col].isin(train_blocks)]
        
        tree = BallTree(coords_rad[train_candidates.index], metric='haversine')
        leaked_indices = tree.query_radius(test_coords, r=buffer_rad)
        flattened_leaks = set([idx for sublist in leaked_indices for idx in sublist])
        
        valid_train_idx = train_candidates.index.difference(train_candidates.index[list(flattened_leaks)])
        folds.append((valid_train_idx.to_numpy(), test_idx.to_numpy()))
    return folds

custom_cv_folds = generate_purged_folds(dataset, 'spatial_block_id', ['Latitude', 'Longitude'], n_splits=5, buffer_km=SPATIAL_BUFFER_KM)

FEATURE_COLS = [
    'slope_deg', 'curvature', 'flow_accumulation', 'soil_clay_pct', 'delta_sar_vv_vh',
    'delta_ndwi', 'monsoon_rainfall_mm', 'grace_gw_storage_cm', 'elevation_m', 'pdf_hydro_zone',
    'distance_to_river_m', 'TWI' # New Features!
]
X = dataset[FEATURE_COLS].copy()
X = X.fillna(X.median())
y = dataset['target'].values
scale_weight = float(np.sum(y == 0) / np.sum(y == 1)) if np.sum(y == 1) > 0 else 1.0
print(f"Feature matrix: {X.shape} | Target vector: {y.shape}")

In [ ]:
print("--- Leak-Free Voting Ensemble with Optuna (Nested Spatial Evaluation) ---")
# To prevent kaggle timeouts, we run a short but effective hyperparameter search (10 trials).
def objective(trial):
    depth = trial.suggest_int('depth', 3, 7)
    lr = trial.suggest_float('lr', 0.01, 0.1, log=True)
    
    xgb_c = xgb.XGBClassifier(n_estimators=100, max_depth=depth, learning_rate=lr, scale_pos_weight=scale_weight, random_state=RANDOM_SEED, verbosity=0)
    cat_c = CatBoostClassifier(iterations=100, depth=depth, learning_rate=lr, scale_pos_weight=scale_weight, random_seed=RANDOM_SEED, verbose=0)
    lgb_c = LGBMClassifier(n_estimators=100, max_depth=depth, learning_rate=lr, class_weight='balanced', random_state=RANDOM_SEED, verbose=-1)
    
    vote = VotingClassifier(estimators=[('xgb', xgb_c), ('cat', cat_c), ('lgb', lgb_c)], voting='soft')
    
    auc_scores = []
    # Evaluate strictly on spatial purged folds
    for train_idx, test_idx in custom_cv_folds:
        vote.fit(X.iloc[train_idx], y[train_idx])
        auc_scores.append(roc_auc_score(y[test_idx], vote.predict_proba(X.iloc[test_idx])[:, 1]))
    return np.mean(auc_scores)

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_SEED))
print("Running Optuna Optimization...")
study.optimize(objective, n_trials=5) # 5 trials is enough to find a strong, safe baseline
best_params = study.best_params
print(f"Best Params Found: {best_params} (CV AUC: {study.best_value:.4f})")

# Final Out-Of-Fold Evaluation using Best Params
all_preds_proba = np.zeros(len(X))
all_y_true = np.zeros(len(X))
final_preds = np.zeros(len(X), dtype=int)

for fold, (train_idx, test_idx) in enumerate(custom_cv_folds):
    xgb_best = xgb.XGBClassifier(n_estimators=150, max_depth=best_params['depth'], learning_rate=best_params['lr'], scale_pos_weight=scale_weight, random_state=RANDOM_SEED, verbosity=0)
    cat_best = CatBoostClassifier(iterations=150, depth=best_params['depth'], learning_rate=best_params['lr'], scale_pos_weight=scale_weight, random_seed=RANDOM_SEED, verbose=0)
    lgb_best = LGBMClassifier(n_estimators=150, max_depth=best_params['depth'], learning_rate=best_params['lr'], class_weight='balanced', random_state=RANDOM_SEED, verbose=-1)
    
    vote_clf = VotingClassifier(estimators=[('xgb', xgb_best), ('cat', cat_best), ('lgb', lgb_best)], voting='soft')
    vote_clf.fit(X.iloc[train_idx], y[train_idx])
    
    fold_proba = vote_clf.predict_proba(X.iloc[test_idx])[:, 1]
    
    all_preds_proba[test_idx] = fold_proba
    all_y_true[test_idx] = y[test_idx]
    final_preds[test_idx] = (fold_proba >= 0.5).astype(int)
    print(f"Fold {fold+1} ROC-AUC: {roc_auc_score(y[test_idx], fold_proba):.4f}")

pr_auc = auc(*precision_recall_curve(all_y_true, all_preds_proba)[1::-1])
print("\n============================================================")
print(f"  UP VOTING ENSEMBLE EVALUATION RESULTS (Buffer={SPATIAL_BUFFER_KM}km)")
print("============================================================")
print(f"  ROC-AUC:           {roc_auc_score(all_y_true, all_preds_proba):.4f}")
print(f"  PR-AUC:            {pr_auc:.4f}")
print(f"  Balanced Accuracy: {balanced_accuracy_score(all_y_true, final_preds):.4f}")
print(f"  MCC:               {matthews_corrcoef(all_y_true, final_preds):.4f}")
print("============================================================")
print(classification_report(all_y_true, final_preds, target_names=['Unsuitable', 'Suitable']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Confusion Matrix
cm = confusion_matrix(all_y_true, final_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix')

# 2. ROC
fpr, tpr, _ = roc_curve(all_y_true, all_preds_proba)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc_score(all_y_true, all_preds_proba):.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
axes[1].set_title('ROC Curve')
axes[1].legend()

# 3. Permutation Importance
# Scikit-learn permutation importance works directly on the VotingClassifier
result = permutation_importance(vote_clf, X.iloc[test_idx], y[test_idx], n_repeats=5, random_state=RANDOM_SEED, scoring='roc_auc')
feat_imp = pd.Series(result.importances_mean, index=FEATURE_COLS).sort_values(ascending=True)
feat_imp.plot(kind='barh', ax=axes[2], color='teal')
axes[2].set_title('Permutation Importance (Last Fold)')

plt.tight_layout()
plt.savefig('/kaggle/working/up_voting_results.png', dpi=150)
plt.show()

In [ ]:
# Final production training on 100% data
xgb_prod = xgb.XGBClassifier(n_estimators=200, max_depth=best_params['depth'], learning_rate=best_params['lr'], scale_pos_weight=scale_weight, random_state=RANDOM_SEED, verbosity=0)
cat_prod = CatBoostClassifier(iterations=200, depth=best_params['depth'], learning_rate=best_params['lr'], scale_pos_weight=scale_weight, random_seed=RANDOM_SEED, verbose=0)
lgb_prod = LGBMClassifier(n_estimators=200, max_depth=best_params['depth'], learning_rate=best_params['lr'], class_weight='balanced', random_state=RANDOM_SEED, verbose=-1)

prod_model = VotingClassifier(estimators=[('xgb', xgb_prod), ('cat', cat_prod), ('lgb', lgb_prod)], voting='soft')
prod_model.fit(X, y)

model_path = f'/kaggle/working/{VERSION_ID}.pkl'
joblib.dump(prod_model, model_path)

# Verify serialization
test_load = joblib.load(model_path)
test_preds = test_load.predict_proba(X.iloc[:2])
print("Artifact verified! Model successfully unpickled and executed prediction.")
print(f'Production Artifact exported: {VERSION_ID}.pkl')